# ray.remote()

@ray.remote 是 Ray 框架中最核心的装饰器，它的作用非常直观：<font color='red'>将普通的 Python 函数或类，瞬间转化为可以在分布式集群中并行执行的“远程实体”。</font>

根据你装饰的对象不同（<font color='green'>函数 vs 类</font>），它的用法和行为会有显著区别。我们可以把它拆解为两种主要模式：
- <font color='green'>无状态的任务（Task）</font>
- <font color='green'>有状态的服务（Actor）</font>

以下是 @ray.remote 的详细介绍和实战指南：

@ray.remote 支持丰富的参数，用于控制任务如何调度。

| 参数 | 说明 | 示例 |
| :--- | :--- | :--- |
| `num_cpus` | 任务所需的 CPU 核数 | `@ray.remote(num_cpus=2)` |
| `num_gpus` | 任务所需的 GPU 数量 | `@ray.remote(num_gpus=0.5)` |
| `memory` | 任务所需的内存大小 (字节) | `@ray.remote(memory=100*1024*1024)` |
| `max_retries` | 任务失败时的自动重试次数 | `@ray.remote(max_retries=3)` |
| `num_returns` | 函数返回值的数量（默认1） | `@ray.remote(num_returns=2)` |



## 模式一：无状态任务 (Remote Functions)

### 适用场景：
- 数据处理、并行计算、一次性任务（类似于 MapReduce 中的 Map）。

当你用 @ray.remote 装饰一个函数时，它变成了一个可以在集群任意节点上异步执行的任务。

In [ ]:
import ray
import time

ray.init()

# 1. 装饰函数
@ray.remote
def heavy_compute(x):
    time.sleep(1) # 模拟耗时操作
    return x * x

# 2. 调用任务 (注意是 .remote())
# 这行代码会立即返回一个 ObjectRef (句柄)，不会等待结果
future = heavy_compute.remote(4)

# 3. 获取结果
# ray.get 是阻塞的，直到任务完成
result = ray.get(future)
print(result) # 输出: 16

### 进阶技巧

#### 并行执行：
- 你可以快速提交大量任务，Ray 会自动调度它们并行运行。

In [ ]:
futures = [heavy_compute(i) for i in range(100)]

# 等待所有任务完成
results = ray.get(futures)

#### 指定资源：
- 如果任务需要 GPU 或大量内存，可以在装饰器中声明。

In [ ]:
@ray.remote(num_gpus=1,num_cpus=2)
def gpu_train():
    # 只有在有GPU的节点上才会执行
    pass

## 模式二：有状态服务 (Actors)

### 适用场景：
- 维护状态（如计数器、缓存）、数据库连接、复杂的类对象（如你提到的 TaskRunner）。

当你用 @ray.remote <font color='red'>装饰一个类时</font>，它变成了一个 Actor。每个 Actor 实例都是一个独立的进程，拥有自己的内存空间和生命周期。

In [ ]:
@ray.remote
class Counter:
    def __init__(self):
        self.value = 0
    def increment(self):
        self.value +=1
        return self.value

# 1. 创建actor实例
# 这会在后台启动一个新的进程

counter_actor = Counter.remote()

# 2. 调用方法
# 注意：这里也是用.remote(),因为是在请求远程进程执行方法
future = counter_actor.increment.remote()

# 3. 获取结果
print(ray.get(future)) # 输出：1
print(ray.get(counter_actor.increment.remote())) # 输出：2（状态被保留了）

###### 关键点
- 状态持久化：Counter 里的 self.value 会一直存在，不会因为方法调用结束而消失。
- 串行执行：默认情况下，同一个 Actor 实例的方法调用是串行执行的（一个接一个），这避免了复杂的锁机制。

###### 最佳实践：尽量批量使用 ray.get，而不是在循环里一个个 get，以最大化并行度。

In [ ]:
# ❌ 效率低：串行等待
for i in range(10):
    result = ray.get(task.remote(i)) 

# ✅ 效率高：并行执行，最后统一等待
futures = [task.remote(i) for i in range(10)]
results = ray.get(futures)

## 常见问题与技巧
- 小任务陷阱：

不要将极小的任务（如简单的加法）变成 remote 任务。因为 Ray 启动进程和调度任务有开销（约几十微秒到毫秒级）。如果任务执行时间小于调度开销，反而会更慢。
建议：将小任务合并成大任务（Batching）。

- 死锁风险：

如果在 Actor 的方法内部调用了 ray.get 等待另一个任务，而那个任务又需要等待当前 Actor 释放资源，就会死锁。

     - 解决：使用 ray.wait 进行异步等待，或者避免在 Actor 内部进行复杂的同步阻塞调用。
       
- 嵌套 Remote：
Remote 函数内部可以调用另一个 Remote 函数。Ray 支持这种嵌套，但要注意资源层级关系。

## 总结

@ray.remote 是 Ray 的魔法开关：

1. 装饰函数 -> 变成并行任务（无状态，用完即走）。
2. 装饰类 -> 变成分布式服务（有状态，长期存活）。
3. 调用方式 -> 必须用 .remote()，返回 ObjectRef。
4. 获取结果 -> 使用 ray.get()